# AI-Based Phishing Website Detection - Model Training & Benchmark Comparison
**Institution:** SRM Institute of Science and Technology  
**Authors:** Daksh Jaiswal (RA2411003030048) & Unnatti Dixit (RA2411003030009)  
**Supervised Models:** Logistic Regression, Decision Tree, Random Forest, Support Vector Machine (SVM)  
---
### Methodology Compliance
Follows Slides 7 & 9 from the presentation deck:
- Pure URL-only features (no DOM / network lookups)
- Stratified 80/20 train/test split (`random_state=42`)
- Scaling ONLY on training set for LR and SVM via `Pipeline`
- 5-fold cross validation with hyperparameter tuning
- Comprehensive test evaluation (Accuracy, Precision, Recall, F1, ROC-AUC, Latency)
- Interpretability and Real False Positive / False Negative Error Analysis


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

results_dir = os.path.join('..', 'results')
metrics_df = pd.read_csv(os.path.join(results_dir, 'metrics.csv'))
print('Trained Model Performance Summary:')
metrics_df[['Model', 'CV_F1_Mean', 'Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1', 'Test_ROC_AUC', 'Inference_Latency_ms']]


## 1. Benchmark Metrics Visualization
Comparing test performance across the 4 classifiers on the identical held-out test set (2,286 samples).


In [ ]:
from IPython.display import Image
Image(filename=os.path.join(results_dir, 'model_comparison_bar.png'))


## 2. ROC Curves & Discrimination Capacity
Receiver Operating Characteristic curves measuring True Positive Rate (Sensitivity) vs False Positive Rate.


In [ ]:
Image(filename=os.path.join(results_dir, 'roc_curves.png'))


## 3. Confusion Matrices & Error Breakdown
True Negatives (TN), False Positives (FP), False Negatives (FN), and True Positives (TP).


In [ ]:
Image(filename=os.path.join(results_dir, 'confusion_matrices.png'))


## 4. Model Explainability & Feature Importance
Examining top features that push URLs towards phishing:


In [ ]:
Image(filename=os.path.join(results_dir, 'feature_importance.png'))


## 5. Real Test Error Analysis
Inspecting genuine False Positives (legitimate sites flagged as phishing) and False Negatives (phishing URLs missed as safe).


In [ ]:
error_df = pd.read_csv(os.path.join(results_dir, 'error_analysis.csv'))
print(f'Total Misclassifications: {len(error_df)}')
print('\nSample False Positives (Legitimate Flagged as Phishing):')
display_cols = ['url', 'error_type', 'phishing_probability']
fps = error_df[error_df['error_type'].str.contains('False Positive')].head(5)
print(fps[display_cols].to_string(index=False))

print('\nSample False Negatives (Phishing Missed as Safe):')
fns = error_df[error_df['error_type'].str.contains('False Negative')].head(5)
print(fns[display_cols].to_string(index=False))


## 6. Best Model Selection & Viva Justification
**Selected Champion Model:** Random Forest

**Justification:**
1. **Superior Phishing Recall (89.41%):** In cybersecurity, False Negatives (missing an active phishing attack) carry severe financial and credential loss consequences. Random Forest achieved the lowest False Negative count (121 out of 1,143).
2. **Highest Overall F1-Score (0.8949) & ROC-AUC (0.9620):** Strongest discrimination boundary across all operating thresholds.
3. **Ultra-Low Inference Latency (0.018 ms/URL):** Capable of scanning >50,000 URLs per second, ideal for inline perimeter network proxies and browser extensions.
4. **No Feature Scaling Required:** Tree-based partitioning naturally handles heterogeneous lexical count, ratio, and binary features without sensitive scaling assumptions.
